Cell 1: Setup & Imports

In [1]:
# Step 2: Build events.csv with two-phase filtering
# Phase A: Collect everything (2019-2024)
# Phase B: Drop only extreme low-coverage firms

import pandas as pd
import requests
import time
from datetime import datetime
import os
from urllib.parse import urljoin

# Create folder structure
os.makedirs("data/02_filings_meta", exist_ok=True)

print("✓ Setup complete!")
print("="*60)

✓ Setup complete!


Cell 2: Load Firm List

In [2]:
# Load your firm list
firms = pd.read_excel("firm_list.xlsx")

# Clean column names
firms.columns = [col.lower().strip() for col in firms.columns]

# Clean data
firms["ticker"] = firms["ticker"].astype(str).str.upper().str.strip()
firms["cik"] = firms["cik"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
firms["cik_padded"] = firms["cik"].str.zfill(10)

print(f"✓ Loaded {len(firms)} firms from firm_list.xlsx")
print(f"\nFirst 5 firms:")
print(firms.head())
print("="*60)

✓ Loaded 148 firms from firm_list.xlsx

First 5 firms:
  ticker      cik  cik_padded
0   AAPL   320193  0000320193
1   ABBV  1551152  0001551152
2    ABT     1800  0000001800
3    ACN  1467373  0001467373
4   ADBE   796343  0000796343


Cell 3: SEC API Helper Functions

In [3]:
# SEC Configuration
HEADERS = {
    "User-Agent": "Cardiff University Student yourname@cardiff.ac.uk"  # ← CHANGE THIS!
}

BASE_URL = "https://data.sec.gov/submissions/"

# ---------------------------------------------------------
# Function 1: Polite JSON requests with retry
# ---------------------------------------------------------
def get_json(url, retries=3, sleep=0.25):
    """
    Download JSON from SEC with retry logic.
    Respects SEC rate limits (10 requests/second).
    """
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=HEADERS, timeout=30)
            response.raise_for_status()
            time.sleep(sleep)  # Rate limit: 0.25s = 4 requests/sec (safe)
            return response.json()
        except Exception as e:
            if attempt == retries - 1:
                raise
            time.sleep(1.0 + attempt)  # Exponential backoff

# ---------------------------------------------------------
# Function 2: Convert SEC JSON to DataFrame
# ---------------------------------------------------------
def filings_to_df(ticker, cik_raw, json_payload):
    """
    Extract 10-K and 10-Q filings from SEC JSON payload.
    Returns DataFrame with one row per filing.
    """
    recent = json_payload.get("filings", {}).get("recent", {})

    forms = recent.get("form", [])
    dates = recent.get("filingDate", [])
    accs = recent.get("accessionNumber", [])

    rows = []

    for form, fdate, acc in zip(forms, dates, accs):
        # Keep ONLY 10-K and 10-Q (no amendments)
        if form in ["10-K", "10-Q"]:
            d = datetime.strptime(fdate, "%Y-%m-%d")

            rows.append({
                "ticker": ticker,
                "cik": cik_raw,
                "filing_date": fdate,
                "filing_type": form,
                "accession_number": acc,
                "year": d.year,
                "quarter": (d.month - 1) // 3 + 1
            })

    return pd.DataFrame(rows)

# ---------------------------------------------------------
# Function 3: Fetch ALL filings (main + older batches)
# ---------------------------------------------------------
def fetch_all_company_filings(ticker, cik_padded, cik_raw):
    """
    CRITICAL FUNCTION:
    Fetches BOTH recent AND historical filings from SEC.

    SEC's main JSON only contains ~1 year of data.
    Older filings are in separate JSON files listed in 'files' array.
    """
    # Step 1: Fetch main submissions JSON
    main_url = f"{BASE_URL}CIK{cik_padded}.json"
    main_json = get_json(main_url)

    # Extract recent filings
    df_list = [filings_to_df(ticker, cik_raw, main_json)]

    # Step 2: Fetch older filing batches (THIS IS KEY!)
    extra_files = main_json.get("filings", {}).get("files", [])

    for file_info in extra_files:
        filename = file_info.get("name")

        if filename:
            # Build full URL for older filings
            extra_url = urljoin(BASE_URL, filename)

            # Fetch and extract
            extra_json = get_json(extra_url)
            df_list.append(filings_to_df(ticker, cik_raw, extra_json))

    # Step 3: Combine all batches
    all_filings = pd.concat(df_list, ignore_index=True)

    # Remove duplicates (rare but possible)
    all_filings = all_filings.drop_duplicates(
        subset=["accession_number", "filing_date", "filing_type"]
    )

    return all_filings

print("✓ Helper functions loaded!")
print("="*60)

✓ Helper functions loaded!


Cell 4: PHASE A - Collect All Data

In [4]:
print("PHASE A: COLLECTING ALL FILINGS")
print("="*60)
print(f"Fetching data for {len(firms)} firms...")
print("This will take approximately 20-30 minutes")
print("="*60)

all_events = []
errors = []

for index, row in firms.iterrows():
    ticker = row["ticker"]
    cik_padded = row["cik_padded"]
    cik_raw = row["cik"]

    try:
        # Fetch all filings for this company
        company_filings = fetch_all_company_filings(ticker, cik_padded, cik_raw)
        all_events.append(company_filings)

        # Progress indicator
        print(f"✓ {index+1:3d}/148: {ticker:6s} → {len(company_filings):3d} filings")

    except Exception as e:
        # Record error but continue
        errors.append({
            "ticker": ticker,
            "cik": cik_padded,
            "error": str(e)
        })
        print(f"✗ {index+1:3d}/148: {ticker:6s} → FAILED: {e}")

print("="*60)
print("✓ Data collection complete!")

# Combine all company data
if all_events:
    events_all = pd.concat(all_events, ignore_index=True)
else:
    events_all = pd.DataFrame()

print(f"\n✓ Total raw filings collected: {len(events_all):,}")
print(f"✓ Successful firms: {len(all_events)}")
print(f"✗ Failed firms: {len(errors)}")

if errors:
    print(f"\nFailed companies:")
    for err in errors[:10]:  # Show first 10
        print(f"  - {err['ticker']}: {err['error'][:60]}")
    if len(errors) > 10:
        print(f"  ... and {len(errors)-10} more")

print("="*60)

PHASE A: COLLECTING ALL FILINGS
Fetching data for 148 firms...
This will take approximately 20-30 minutes
✓   1/148: AAPL   →  44 filings
✓   2/148: ABBV   →  37 filings
✓   3/148: ABT    →  32 filings
✓   4/148: ACN    →  12 filings
✓   5/148: ADBE   →  34 filings
✓   6/148: ADI    →  32 filings
✓   7/148: ADP    →  33 filings
✓   8/148: AMAT   →  54 filings
✓   9/148: AMD    →  37 filings
✓  10/148: AMGN   →  34 filings
✓  11/148: AMT    →  37 filings
✓  12/148: AMZN   →  26 filings
✓  13/148: ANET   →  19 filings
✓  14/148: APH    →  49 filings
✓  15/148: APP    →  20 filings
✓  16/148: AVGO   →  31 filings
✓  17/148: AXP    →  29 filings
✓  18/148: BA     →  29 filings
✓  19/148: BAC    →   4 filings
✓  20/148: BKNG   →  48 filings
✓  21/148: BLK    →   4 filings
✓  22/148: BMY    →  29 filings
✓  23/148: BRK.B  →  39 filings
✓  24/148: BSX    →  24 filings
✓  25/148: BX     →  37 filings
✓  26/148: C      →   4 filings
✓  27/148: CAT    →  31 filings
✓  28/148: CB     →  34 filing

Cell 5: Filter to Time Window

In [5]:
print("\nPHASE A: FILTERING TO TIME WINDOW")
print("="*60)

# Define time window
START_DATE = "2019-01-01"
END_DATE = "2024-12-31"

print(f"Time window: {START_DATE} to {END_DATE}")

# Convert to datetime
events_all["filing_date"] = pd.to_datetime(events_all["filing_date"])

# Filter to time window
events_all = events_all[
    (events_all["filing_date"] >= START_DATE) &
    (events_all["filing_date"] <= END_DATE)
].copy()

# Sort chronologically
events_all = events_all.sort_values(
    ["ticker", "filing_date"]
).reset_index(drop=True)

print(f"\n✓ Filings after time filter: {len(events_all):,}")
print(f"✓ Date range: {events_all['filing_date'].min().date()} to {events_all['filing_date'].max().date()}")
print(f"\n✓ Filing type breakdown:")
print(events_all["filing_type"].value_counts())

# Save events_all (before cleaning)
events_all.to_csv("data/02_filings_meta/events_all.csv", index=False)
print(f"\n✓ Saved: data/02_filings_meta/events_all.csv")
print("="*60)


PHASE A: FILTERING TO TIME WINDOW
Time window: 2019-01-01 to 2024-12-31

✓ Filings after time filter: 3,021
✓ Date range: 2019-01-08 to 2024-12-20

✓ Filing type breakdown:
filing_type
10-Q    2280
10-K     741
Name: count, dtype: int64

✓ Saved: data/02_filings_meta/events_all.csv


Cell 6: PHASE B - Identify Low-Coverage Firms

In [6]:
print("\nPHASE B: IDENTIFYING LOW-COVERAGE FIRMS")
print("="*60)

# Count filings per firm by type
filings_count = events_all.pivot_table(
    index="ticker",
    columns="filing_type",
    values="filing_date",
    aggfunc="count",
    fill_value=0
)

# Add total count
filings_count["total_filings"] = filings_count.sum(axis=1)

# Rename columns for clarity
filings_count.columns = ["tenK_count", "tenQ_count", "total_filings"]

print("Coverage statistics across all firms:")
print(filings_count.describe())

print("\n" + "="*60)
print("APPLYING MINIMUM COVERAGE RULE")
print("="*60)
print("Rule: Drop firms with:")
print("  - total_filings < 10  OR")
print("  - tenK_count < 2")
print("="*60)

# Identify firms to KEEP
firms_to_keep = filings_count[
    (filings_count["total_filings"] >= 10) &
    (filings_count["tenK_count"] >= 2)
]

# Identify firms to DROP
firms_to_drop = filings_count[
    (filings_count["total_filings"] < 10) |
    (filings_count["tenK_count"] < 2)
]

print(f"\n✓ Firms KEEPING: {len(firms_to_keep)}")
print(f"✗ Firms DROPPING: {len(firms_to_drop)}")

if len(firms_to_drop) > 0:
    print(f"\nFirms being EXCLUDED (extreme low coverage):")
    print(firms_to_drop.sort_values("total_filings"))
else:
    print("\n✓ No firms to exclude! All have sufficient coverage.")

print("="*60)


PHASE B: IDENTIFYING LOW-COVERAGE FIRMS
Coverage statistics across all firms:
       tenK_count  tenQ_count  total_filings
count  142.000000  142.000000     142.000000
mean     5.218310   16.056338      21.274648
std      1.399677    3.735539       5.113110
min      0.000000    3.000000       3.000000
25%      5.000000   15.000000      20.250000
50%      6.000000   18.000000      24.000000
75%      6.000000   18.000000      24.000000
max      6.000000   18.000000      24.000000

APPLYING MINIMUM COVERAGE RULE
Rule: Drop firms with:
  - total_filings < 10  OR
  - tenK_count < 2

✓ Firms KEEPING: 132
✗ Firms DROPPING: 10

Firms being EXCLUDED (extreme low coverage):
        tenK_count  tenQ_count  total_filings
ticker                                       
META             0           3              3
CRH              1           3              4
WFC              1           3              4
VZ               1           5              6
CRM              1           5              6
NFLX

Cell 7: Create Cleaned Dataset

In [7]:
print("\nPHASE B: CREATING CLEANED DATASET")
print("="*60)

# Keep only firms with sufficient coverage
keep_tickers = firms_to_keep.index.tolist()
events_clean = events_all[events_all["ticker"].isin(keep_tickers)].copy()

print(f"✓ Firms in cleaned dataset: {events_clean['ticker'].nunique()}")
print(f"✓ Total filings in cleaned dataset: {len(events_clean):,}")
print(f"✓ Date range: {events_clean['filing_date'].min().date()} to {events_clean['filing_date'].max().date()}")

print(f"\n✓ Filing type distribution:")
print(events_clean["filing_type"].value_counts())

print(f"\n✓ Filings per company statistics:")
company_counts = events_clean.groupby("ticker").size()
print(f"  Mean: {company_counts.mean():.1f}")
print(f"  Median: {company_counts.median():.0f}")
print(f"  Min: {company_counts.min()}")
print(f"  Max: {company_counts.max()}")
print(f"  Std: {company_counts.std():.1f}")

# Save cleaned dataset
events_clean.to_csv("data/02_filings_meta/events_clean.csv", index=False)
print(f"\n✓ Saved: data/02_filings_meta/events_clean.csv")

# Save excluded firms documentation
if len(firms_to_drop) > 0:
    excluded_df = pd.DataFrame({
        "ticker": firms_to_drop.index,
        "tenK_count": firms_to_drop["tenK_count"],
        "tenQ_count": firms_to_drop["tenQ_count"],
        "total_filings": firms_to_drop["total_filings"],
        "exclusion_reason": "Insufficient filing coverage (total < 10 OR 10-K < 2)"
    })
    excluded_df.to_csv("data/02_filings_meta/excluded_firms.csv", index=False)
    print(f"✓ Saved: data/02_filings_meta/excluded_firms.csv")

print("="*60)


PHASE B: CREATING CLEANED DATASET
✓ Firms in cleaned dataset: 132
✓ Total filings in cleaned dataset: 2,955
✓ Date range: 2019-01-08 to 2024-12-20

✓ Filing type distribution:
filing_type
10-Q    2228
10-K     727
Name: count, dtype: int64

✓ Filings per company statistics:
  Mean: 22.4
  Median: 24
  Min: 10
  Max: 24
  Std: 3.2

✓ Saved: data/02_filings_meta/events_clean.csv
✓ Saved: data/02_filings_meta/excluded_firms.csv


Cell 8: Final Summary & Validation

In [14]:
print("\n" + "="*60)
print("STEP 2 COMPLETE - FINAL SUMMARY")
print("="*60)

print(f"\n DATASET OVERVIEW:")
print(f"  Starting firms: 148")
print(f"  Successful data collection: {len(all_events)}")
print(f"  Failed data collection: {len(errors)}")
print(f"  Firms in events_all.csv: {events_all['ticker'].nunique()}")
print(f"  Firms in events_clean.csv: {events_clean['ticker'].nunique()}")
print(f"  Firms excluded: {len(firms_to_drop)}")

print(f"\n FILINGS COUNT:")
print(f"  events_all.csv: {len(events_all):,} filings")
print(f"  events_clean.csv: {len(events_clean):,} filings")

print(f"\n TIME COVERAGE:")
print(f"  Start date: {events_clean['filing_date'].min().date()}")
print(f"  End date: {events_clean['filing_date'].max().date()}")
print(f"  Span: {(events_clean['filing_date'].max() - events_clean['filing_date'].min()).days} days")

print(f"\n FILING TYPES:")
for ftype, count in events_clean["filing_type"].value_counts().items():
    print(f"  {ftype}: {count:,}")

print(f"\n FILES CREATED:")
print(f"  ✓ data/02_filings_meta/events_all.csv")
print(f"  ✓ data/02_filings_meta/events_clean.csv")
if len(firms_to_drop) > 0:
    print(f"  ✓ data/02_filings_meta/excluded_firms.csv")

print(f"\n READY FOR STEP 3: MD&A Extraction")
print("="*60)

# Download files from Colab
from google.colab import files

print("\nDownloading files...")
files.download("data/02_filings_meta/events_all.csv")
files.download("data/02_filings_meta/events_clean.csv")
if len(firms_to_drop) > 0:
    files.download("data/02_filings_meta/excluded_firms.csv")

print("\n✓ All files downloaded!")



STEP 2 COMPLETE - FINAL SUMMARY

 DATASET OVERVIEW:
  Starting firms: 148
  Successful data collection: 148
  Failed data collection: 0
  Firms in events_all.csv: 142
  Firms in events_clean.csv: 132
  Firms excluded: 10

 FILINGS COUNT:
  events_all.csv: 3,021 filings
  events_clean.csv: 2,955 filings

 TIME COVERAGE:
  Start date: 2019-01-08
  End date: 2024-12-20
  Span: 2173 days

 FILING TYPES:
  10-Q: 2,228
  10-K: 727

 FILES CREATED:
  ✓ data/02_filings_meta/events_all.csv
  ✓ data/02_filings_meta/events_clean.csv
  ✓ data/02_filings_meta/excluded_firms.csv

 READY FOR STEP 3: MD&A Extraction



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ All files downloaded!


In [18]:
from google.colab import files

files.download("data/02_filings_meta/events_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>